# 1: Setup & Installationen

In [ ]:
# --- Google Drive Mount und Projektpfad ---
try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os

    # !!! WICHTIG: Pfad zu deinem Projektordner in Google Drive anpassen !!!
    #             (Der Ordner, wo RPG.ipynb und die .py Dateien liegen)
    project_path = '/content/drive/MyDrive/Colab Notebooks/' # Beispielpfad, ANPASSEN!

    # Versuche, in das Projektverzeichnis zu wechseln
    os.chdir(project_path)
    print(f"Aktuelles Arbeitsverzeichnis: {os.getcwd()}")

except ImportError:
    print("Google Colab Umgebung nicht erkannt. Arbeite im aktuellen Verzeichnis.")
    project_path = os.getcwd()
    print(f"Aktuelles Arbeitsverzeichnis: {project_path}")
except FileNotFoundError:
    print(f"FEHLER: Projektpfad '{project_path}' nicht gefunden. Bitte Pfad prüfen.")
    # Fallback auf aktuelles Verzeichnis
    project_path = os.getcwd()
    print(f"Fallback Arbeitsverzeichnis: {project_path}")
except Exception as e:
    print(f"Fehler beim Wechseln des Verzeichnisses: {e}")
    project_path = os.getcwd() # Fallback
    print(f"Fallback Arbeitsverzeichnis: {project_path}")


# --- Installationen ---
print("\nInstalliere/Überprüfe Abhängigkeiten...")
# ipywidgets für HTML-Anzeige etc.
!pip install ipywidgets -q
# Gymnasium für RL Umgebung
!pip install gymnasium -q
# Stable-Baselines3 für RL Algorithmen
!pip install stable-baselines3[extra] -q # [extra] für TensorBoard etc.
print("Abhängigkeiten installiert/überprüft.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Aktuelles Arbeitsverzeichnis: /content/drive/MyDrive/Colab Notebooks

Installiere/Überprüfe Abhängigkeiten...
Abhängigkeiten installiert/überprüft.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Aktuelles Arbeitsverzeichnis: /content/drive/MyDrive/Colab Notebooks

Installiere/Überprüfe Abhängigkeiten...
Abhängigkeiten installiert/überprüft.


# 2: Basis-Imports & Modul-Imports

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 2: Basis-Imports & Modul-Imports (V5 - Mit Unmount/Remount & Modul-Löschen)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print("\nImportiere Basis-Bibliotheken und eigene Module...")
import os
import sys
import random
import time
import math
import datetime
import numpy as np
from collections import defaultdict
from typing import Optional, Dict, List, Union, Tuple, Any
import traceback
import importlib # Bleibt wichtig für den Import

# Wichtig für HTML UI und Fortschrittsanzeige
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# --- 1. Google Drive Unmount/Mount ---
try:
    from google.colab import drive
    # --- VERSUCHE ZUERST ZU UNMOUNTEN ---
    print("\nVersuche Google Drive zu unmounten (falls bereits gemountet)...")
    try:
        drive.flush_and_unmount()
        print("Drive erfolgreich unmounted.")
    except Exception as e:
        # Fehler hier ist oft okay, wenn Drive nicht gemountet war
        print(f"  Info/Fehler beim Unmounten: {e} (Kann ignoriert werden, wenn Drive nicht gemountet war)")
    # --- ENDE UNMOUNT ---

    # --- JETZT NEU MOUNTEN ---
    print("\nVersuche Google Drive neu zu mounten...")
    drive.mount('/content/drive', force_remount=True) # force_remount ist hier wichtig
    print("Google Drive gemountet.")
    # --- ENDE MOUNT ---

    # !!! WICHTIG: Pfad zu deinem Projektordner in Google Drive anpassen !!!
    project_path = '/content/drive/MyDrive/Colab Notebooks/' # Beispielpfad, ANPASSEN!

    # Überprüfe, ob der Pfad existiert
    if not os.path.isdir(project_path):
        print(f"FEHLER: Projektpfad '{project_path}' nicht gefunden. Bitte Pfad prüfen.")
        print(f"Verwende aktuelles Verzeichnis als Fallback: {os.getcwd()}")
        project_path = os.getcwd() # Fallback, wenn der Pfad nicht stimmt
    else:
        print(f"Projektpfad gesetzt auf: {project_path}")
        # Optional: Ins Verzeichnis wechseln (kann manchmal helfen)
        try:
            os.chdir(project_path)
            print(f"Arbeitsverzeichnis geändert zu: {os.getcwd()}")
        except Exception as e:
            print(f"Fehler beim Wechseln des Verzeichnisses: {e}")


except ImportError:
    print("Google Colab Umgebung nicht erkannt. Arbeite im aktuellen Verzeichnis.")
    project_path = os.getcwd()
    print(f"Aktuelles Arbeitsverzeichnis: {project_path}")
except Exception as e:
    print(f"Genereller Fehler beim Mounten/Unmounten von Google Drive: {e}")
    project_path = os.getcwd() # Fallback
    print(f"Fallback Arbeitsverzeichnis: {project_path}")


# --- 2. Hilfsfunktion zum Prüfen des Modul-Zeitstempels ---
def check_module_timestamp(module_name: str, module_obj: Optional[Any]) -> None:
    """ Prüft, ob ein Modul geladen wurde und gibt dessen Änderungsdatum aus. """
    if module_obj is None:
        print(f"FEHLER: Modul '{module_name}' konnte nicht importiert werden.")
        return
    try:
        file_path = getattr(module_obj, '__file__', None)
        if file_path and os.path.exists(file_path):
            timestamp = os.path.getmtime(file_path)
            mod_time = datetime.datetime.fromtimestamp(timestamp)
            print(f"  -> Modul '{module_name}' Info: Letzte Änderung: {mod_time.strftime('%Y-%m-%d %H:%M:%S')} (Datei: {os.path.basename(file_path)})")
        elif file_path:
            print(f"  -> Modul '{module_name}' geladen, aber Datei nicht gefunden: {file_path}")
        else:
            print(f"  -> Modul '{module_name}' geladen, aber Pfad (__file__) nicht verfügbar.")
    except Exception as e:
        print(f"  -> Fehler beim Abrufen des Zeitstempels für Modul '{module_name}': {e}")

# --- 3. Eigene Module importieren und prüfen ---
# Füge Projektpfad zu sys.path hinzu, falls noch nicht geschehen
if project_path not in sys.path:
    sys.path.insert(0, project_path)
    print(f"Projektpfad '{project_path}' zu sys.path hinzugefügt.")

# Liste der zu importierenden Module
module_names = [
    'rpg_config',
    'rpg_definitions',
    'rpg_game_logic',
    'rpg_ui',
    'rpg_env',
    'rpg_training_utils'
]
imported_modules = {}

print("\nVersuche, eigene Module zu importieren (mit Cache-Löschung) und Zeitstempel zu prüfen:")
all_imports_successful = True

for name in module_names:
    print(f"\nVerarbeite Modul: '{name}'...")
    # Explizites Löschen aus sys.modules
    if name in sys.modules:
        print(f"  - Entferne altes Modul '{name}' aus sys.modules...")
        try:
            del sys.modules[name]
            print(f"  - Modul '{name}' erfolgreich entfernt.")
        except KeyError:
            print(f"  - Warnung: Modul '{name}' war bereits aus sys.modules entfernt?")
    else:
        print(f"  - Modul '{name}' war nicht in sys.modules.")

    module = None
    try:
        # Importiere das Modul frisch
        module = importlib.import_module(name)
        print(f"  - Modul '{name}' neu importiert.")
        imported_modules[name] = module # Speichere das Modulobjekt
        # Prüfe den Zeitstempel direkt nach dem Import
        check_module_timestamp(name, module)
    except ImportError as e:
        print(f"\nFEHLER: Konnte Modul '{name}' nicht importieren: {e}")
        print("  -> Stelle sicher, dass die .py Datei existiert und keine Syntaxfehler enthält.")
        print("  -> Überprüfe ggf. Abhängigkeiten innerhalb des Moduls.")
        imported_modules[name] = None # Markiere als fehlgeschlagen
        all_imports_successful = False
    except Exception as e:
        print(f"\nFEHLER: Unerwarteter Fehler beim Import/Prüfen von Modul '{name}': {e}")
        traceback.print_exc()
        imported_modules[name] = None
        all_imports_successful = False

if all_imports_successful:
     print("\nAlle eigenen Module erfolgreich importiert/neu geladen und geprüft.")
else:
     print("\nWARNUNG: Mindestens ein Modul konnte nicht korrekt importiert werden. Folgefehler wahrscheinlich.")
     # Optional: Hier abbrechen?
     # raise ImportError("Ein oder mehrere Module konnten nicht geladen werden.")

# Versuche, die Module direkt in den globalen Namespace zu bringen
try:
    rpg_config = imported_modules.get('rpg_config')
    rpg_definitions = imported_modules.get('rpg_definitions')
    rpg_game_logic = imported_modules.get('rpg_game_logic')
    rpg_ui = imported_modules.get('rpg_ui')
    rpg_env = imported_modules.get('rpg_env')
    rpg_training_utils = imported_modules.get('rpg_training_utils')
    print("Module globalen Variablen zugewiesen.")
except Exception as e:
    print(f"Fehler beim Zuweisen der Module zu globalen Variablen: {e}")


# --- 4. RL Bibliotheken Imports ---
try:
    import gymnasium as gym
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_checker import check_env
    print("\nRL-Bibliotheken importiert.")
except ImportError as e:
    print(f"\nFEHLER: RL-Bibliotheken nicht gefunden: {e}")
    print("Stelle sicher, dass die Installation in Zelle 1 erfolgreich war.")
    raise e

print("\n--- Ende Zelle 2 ---")


Importiere Basis-Bibliotheken und eigene Module...

Versuche Google Drive zu unmounten (falls bereits gemountet)...
Drive erfolgreich unmounted.

Versuche Google Drive neu zu mounten...
Mounted at /content/drive
Google Drive gemountet.
Projektpfad gesetzt auf: /content/drive/MyDrive/Colab Notebooks/
Arbeitsverzeichnis geändert zu: /content/drive/MyDrive/Colab Notebooks

Versuche, eigene Module zu importieren (mit Cache-Löschung) und Zeitstempel zu prüfen:

Verarbeite Modul: 'rpg_config'...
  - Entferne altes Modul 'rpg_config' aus sys.modules...
  - Modul 'rpg_config' erfolgreich entfernt.
Konfigurationsdatei rpg_config.py geschrieben/überschrieben (V4 - Ordner umbenannt).
  - LOAD_EXISTING_MODELS: False
  - Log Verzeichnis: /content/drive/MyDrive/Colab Notebooks/logs/ppo_klassen_lvl_v2/
  - Modell Verzeichnis: /content/drive/MyDrive/Colab Notebooks/models/klassen_agenten_lvl_v2/
  - Bericht Verzeichnis: /content/drive/MyDrive/Colab Notebooks/Auswertungsberichte/
  - Modul 'rpg_config

In [ ]:
print("Keys in geladenem rpg_definitions.CHAR_PARAMS:", list(rpg_definitions.CHAR_PARAMS.keys()))

Keys in geladenem rpg_definitions.CHAR_PARAMS: ['Krieger', 'Magier', 'Schurke', 'Kleriker', 'Goblin', 'Skelett', 'Ork']


# 3: Initialisierung der Spieldaten (Phase 3 - Objekte erstellen)


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 3: Initialisierung der Spieldaten (Angepasst, V2)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import traceback # Sicherstellen, dass traceback für except Block verfügbar ist

print("\nInitialisiere Spieldaten (Skills)...")

# Stelle sicher, dass Module geladen sind
if 'rpg_definitions' not in locals() or 'rpg_game_logic' not in locals():
     raise ImportError("Benötigte Module (rpg_definitions, rpg_game_logic) nicht geladen. Bitte Zelle 2 ausführen.")

INITIALIZED_SKILLS: Dict[str, rpg_game_logic.Skill] = {}

def initialize_game_data(skill_params_dict: dict) -> Dict[str, rpg_game_logic.Skill]:
    """
    Erstellt Skill-Objekte aus den Parameter-Dictionaries.
    """
    initialized_skills = {}
    print(f"Initialisiere {len(skill_params_dict)} Skills...")
    for skill_name, params in skill_params_dict.items():
        try:
            # Erstelle Skill-Objekt mit den Parametern aus dem Dictionary
            skill_obj = rpg_game_logic.Skill(**params)
            initialized_skills[skill_name] = skill_obj
            # print(f"  - Skill '{skill_name}' initialisiert.")
        except TypeError as e:
            print(f"FEHLER bei Initialisierung von Skill '{skill_name}': {e}")
            print(f"  -> Übergebene Parameter: {params}")
        except Exception as e:
            print(f"Allgemeiner FEHLER bei Initialisierung von Skill '{skill_name}': {e}")
            traceback.print_exc()
    print(f"{len(initialized_skills)} Skills erfolgreich initialisiert.")
    return initialized_skills

# Führe die Initialisierung durch
try:
    INITIALIZED_SKILLS = initialize_game_data(rpg_definitions.SKILL_PARAMS)

    # Überprüfe kurz, ob wichtige Definitionen geladen wurden (angepasst)
    print("Prüfe wichtige Definitionen...")
    definitions_ok = True
    if not getattr(rpg_definitions, 'CHAR_PARAMS', None): # Sicherer Zugriff
        print("WARNUNG: CHAR_PARAMS aus rpg_definitions ist leer oder nicht vorhanden!")
        definitions_ok = False
    # Prüfe auf ALL_SKILLS_MAP statt SKILLS_BY_CLASS
    if not getattr(rpg_definitions, 'ALL_SKILLS_MAP', None): # Sicherer Zugriff
        print("WARNUNG: ALL_SKILLS_MAP aus rpg_definitions ist leer oder nicht vorhanden!")
        definitions_ok = False
    if not getattr(rpg_definitions, 'ALL_BUFFS_DEBUFFS_NAMES', None): # Sicherer Zugriff
         print("WARNUNG: ALL_BUFFS_DEBUFFS_NAMES aus rpg_definitions ist leer oder nicht vorhanden!")
         definitions_ok = False

    if definitions_ok:
        print("Wichtige Definitionen scheinen vorhanden zu sein.")
    else:
        print("-> Mindestens eine wichtige Definition fehlt in rpg_definitions.py!")

    print("\nSpieldaten initialisiert.")

except AttributeError as e:
    # Dieser Fehler sollte jetzt nicht mehr wegen SKILLS_BY_CLASS auftreten
    print(f"\nFEHLER (AttributeError) bei Initialisierung: {e}")
    print("Stelle sicher, dass rpg_definitions.py und rpg_game_logic.py korrekt sind.")
    traceback.print_exc()
except Exception as e:
     print(f"\nFEHLER bei der Initialisierung der Spieldaten: {e}")
     traceback.print_exc()


Initialisiere Spieldaten (Skills)...
Initialisiere 17 Skills...
17 Skills erfolgreich initialisiert.
Prüfe wichtige Definitionen...
Wichtige Definitionen scheinen vorhanden zu sein.

Spieldaten initialisiert.


# 4: RL Umgebung Instanziieren & Prüfen

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 4: RL Umgebung Instanziieren & Prüfen (Angepasst für RPGEnv V6/V7/V8)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print("\nErstelle und prüfe die RL-Umgebung...")

# Stelle sicher, dass Module geladen sind (aus Zelle 2)
# Füge hier einen expliziten traceback Import hinzu, falls er gebraucht wird
import traceback
if 'rpg_env' not in locals() or 'rpg_definitions' not in locals():
    raise ImportError("Benötigte Module (rpg_env, rpg_definitions) nicht geladen. Bitte Zelle 2 ausführen.")

rpg_environment_instance = None # Vorinitialisieren

try:
    # Erstelle eine Instanz der Umgebung
    # NUR noch char_param_definitions und optionale Parameter übergeben!
    print("--> Zelle 4: Aufruf RPGEnv(...) beginnt...")
    rpg_environment_instance = rpg_env.RPGEnv(
        char_param_definitions=rpg_definitions.CHAR_PARAMS,
        # skills_by_class=... <-- Nicht mehr benötigt, entfernt
        # initialized_skills=... <-- Nicht mehr benötigt, entfernt
        all_buffs_debuffs_names=rpg_definitions.ALL_BUFFS_DEBUFFS_NAMES,
        max_buff_duration=rpg_definitions.MAX_BUFF_DURATION,
        max_stacks=rpg_definitions.MAX_STACKS,
        render_mode='ansi' # oder None oder 'human' (falls UI anders gehandhabt wird)
    )
    print("--> Zelle 4: Aufruf RPGEnv(...) beendet.")
    print("RPGEnv Instanz erfolgreich erstellt.")

    # --- Umgebungs-Check (Optional, aber empfohlen) ---
    # Prüft, ob die Umgebung der Gymnasium API entspricht.
    print("Führe Gymnasium Env Check durch...")
    try:
        # Stelle sicher, dass check_env importiert wurde (normalerweise in Zelle 2)
        if 'check_env' not in locals():
            from stable_baselines3.common.env_checker import check_env
            print("Hinweis: stable_baselines3.common.env_checker.check_env nachimportiert.")

        check_env(rpg_environment_instance, warn=True)
        print(">>> Gymnasium Env Check erfolgreich abgeschlossen.")
    except NameError:
        print("FEHLER: check_env nicht gefunden. Konnte Env Check nicht durchführen.")
        print("        Stelle sicher, dass Zelle 2 gelaufen ist und stable_baselines3 importiert wurde.")
    except Exception as e:
        print(f"\nFEHLER beim Gymnasium Env Check: {e}")
        print("------------------------------------------------------------")
        print("Mögliche Ursachen:")
        print("- Inkonsistenzen zwischen Action/Observation Space Definition und step()/reset() Rückgaben.")
        print("- Falsche Datentypen (z.B. nicht numpy array, falscher dtype).")
        print("- Fehler in der reset() oder step() Logik der Umgebung.")
        print("Details:")
        traceback.print_exc() # Behalte Traceback hier bei
        print("------------------------------------------------------------")
        # Umgebung trotzdem behalten, um ggf. manuell zu debuggen

except AttributeError as e:
    # Dieser Fehler sollte jetzt nicht mehr auftreten, da SKILLS_BY_CLASS nicht mehr verwendet wird
    print(f"\nFEHLER: Benötigte Definitionen für RPGEnv nicht gefunden (z.B. aus rpg_definitions): {e}")
    traceback.print_exc() # Zeige den Traceback
    # raise e # Ggf. Fehler weiterwerfen, um Stopp zu erzwingen
except Exception as e:
     print(f"\nFEHLER beim Erstellen oder Prüfen der RPGEnv Instanz: {e}")
     traceback.print_exc()
     # raise e # Ggf. Fehler weiterwerfen


Erstelle und prüfe die RL-Umgebung...
--> Zelle 4: Aufruf RPGEnv(...) beginnt...
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
--> Zelle 4: Aufruf RPGEnv(...) beendet.
RPGEnv Instanz erfolgreich erstellt.
Führe Gymnasium Env Check durch...
Kleriker_Hero benutzt 'Leichte Heilung' auf Kleriker_Hero.
Kleriker_Hero wird um 60 HP geheilt.
Skelett Magier_L1_237 benutzt 'Frostpfeil' auf Kleriker_Hero.
Kleriker_Hero erleidet 13 Schaden.
Ork_L1_874 benutzt 'Wuchtschlag' auf Magier_Hero.
Magier_Hero erleidet 49 Schaden.
Magier_Hero benutzt 'Feuerball' auf Ork_L1_874.
Ork_L1_874 erleidet 41 Schaden.
Ork_L1_874 benutzt 'Wuchtschlag' auf Magier_Hero.
Magier_Hero erleidet 49 Schaden.
Magier_Hero wurde besiegt!
Goblin Krieger_L1_250 benutzt 'Keulenschlag' auf Kleriker_Hero.
Kleriker_Hero erleidet 14 Schaden.
Kleriker_Hero benutzt 'Regeneration' auf Kleriker_Hero.
Effekt 'Regeneration' wird auf Kleriker_Hero angewendet (Dauer: 3, Mods: {'dot_heal': 10}).
Kleriker_Hero 

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 4.1: Funktion zum Anzeigen der Umgebungsinformationen
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# Importiere die Klasse, um den Typ-Hint zu ermöglichen (optional aber gut)
try:
    from rpg_env import RPGEnv
except ImportError:
    class RPGEnv: pass # Dummy für den Fall, dass etwas schiefgeht

def print_env_details(env: RPGEnv):
    """ Gibt detaillierte Informationen über die Action/Observation Spaces der Env aus. """
    if not isinstance(env, RPGEnv):
        print("FEHLER: Übergebenes Objekt ist keine RPGEnv Instanz.")
        return

    print("\nRPG Umgebung Details:")
    print("-" * 50)

    # Action Space Details
    try:
        action_size = env.action_space.n
        # Verwende die action_to_name_map für die Ausgabe
        action_map = getattr(env, 'action_to_name_map', {})
        skills = [name for idx, name in action_map.items() if name not in ["Passen", "Basis-Angriff"]]
        num_skills = len(skills)
        pass_idx = getattr(env, '_pass_action_idx', 'N/A')
        basic_idx = getattr(env, '_basic_attack_idx', 'N/A')

        print(f"=> Aktionsmöglichkeiten (Action Space - Größe: {action_size}):")
        if skills:
             print(f"   - Skills (Indizes 0-{num_skills-1}): {skills}")
        else:
             print("   - Keine Skills gefunden.")
        print(f"   - Passen (Index {pass_idx})")
        print(f"   - Basis-Angriff (Index {basic_idx})")

    except AttributeError as e:
        print(f"Fehler beim Abrufen der Action Space Details: {e}")

    # Observation Space Details
    print("\n=> Beobachtungen (Observation Space - Größe: {}):".format(env.observation_space.shape[0] if hasattr(env.observation_space, 'shape') else 'Unbekannt'))
    try:
        num_res = 8
        num_cd = getattr(env, '_num_skills', len(getattr(env, 'all_skill_objects_list', [])))
        num_buff_features = 0
        buff_list = getattr(env, 'buff_list_for_obs', [])
        if buff_list:
             num_buff_features = len(buff_list) * 2 * 2

        print(f"   - {num_res} Werte: Ressourcen % [0..1] (HP, Mana, Stam, Energy für Held & Gegner)")
        print(f"   - {num_cd} Werte: Skill Cooldowns % [0..1] (für Held, Reihenfolge wie Skill-Liste)")
        print(f"   - {num_buff_features} Werte: Buffs/Debuffs [0..1] (Dauer % & Stacks % für Held & Gegner)")
        if buff_list:
            print(f"     -> Für Effekte: {buff_list}")
        else:
             print("     -> Keine Buff-/Debuff-Effekte im Observation Space definiert.")

    except AttributeError as e:
        print(f"Fehler beim Abrufen der Observation Space Details: {e}")

    print("-" * 50)

# Diese Zelle definiert nur die Funktion, sie wird hier noch nicht aufgerufen.
print("Funktion 'print_env_details' definiert.")

Funktion 'print_env_details' definiert.


# 5: Training der klassenspezifischen Agenten


In [ ]:
# -*- coding: utf-8 -*-
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 5: Training der klassenspezifischen Agenten mit IPyWidget Callback
#          (Angepasst: Debug-Ausgaben in separates Output-Widget umgeleitet)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# --- Erforderliche Imports ---
import os
import numpy as np
import pandas as pd
import datetime
import traceback
from stable_baselines3.common.results_plotter import load_results
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import importlib
import time
import sys # Für sys.modules Check
# *** HIER DIE ÄNDERUNG: Explizite Widget-Imports ***
import ipywidgets as widgets
from IPython.display import display

# --- Stelle sicher, dass Module aus vorherigen Zellen geladen sind ---
try:
    # Überprüfe, ob die notwendigen Module und Variablen existieren
    if 'rpg_config' not in globals(): raise NameError("Modul 'rpg_config' nicht gefunden.")
    if 'rpg_definitions' not in globals(): raise NameError("Modul 'rpg_definitions' nicht gefunden.")
    if 'rpg_env' not in globals(): raise NameError("Modul 'rpg_env' nicht gefunden.")
    if 'rpg_training_utils' not in globals(): raise NameError("Modul 'rpg_training_utils' nicht gefunden.")

    # Überprüfe, ob die benötigten Klassen/Variablen/Funktionen existieren
    if not hasattr(rpg_env, 'RPGEnv'): raise AttributeError("Klasse 'RPGEnv' nicht in Modul 'rpg_env' gefunden.")
    if not hasattr(rpg_definitions, 'CHAR_PARAMS'): raise AttributeError("Variable 'CHAR_PARAMS' nicht in Modul 'rpg_definitions' gefunden.")
    if not hasattr(rpg_config, 'LOG_DIR_BASE'): raise AttributeError("Variable 'LOG_DIR_BASE' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'SUMMARY_SAVE_DIR'): raise AttributeError("Variable 'SUMMARY_SAVE_DIR' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'MODEL_DIR_BASE'): raise AttributeError("Variable 'MODEL_DIR_BASE' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'TOTAL_TIMESTEPS_PER_CLASS'): raise AttributeError("Variable 'TOTAL_TIMESTEPS_PER_CLASS' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'CLASSES_TO_TRAIN'): raise AttributeError("Variable 'CLASSES_TO_TRAIN' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'LOAD_EXISTING_MODELS'): raise AttributeError("Variable 'LOAD_EXISTING_MODELS' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'DEFAULT_PPO_PARAMS'): raise AttributeError("Variable 'DEFAULT_PPO_PARAMS' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_training_utils, 'generate_layperson_log'):
        raise NameError("Funktion 'generate_layperson_log' NICHT in 'rpg_training_utils' gefunden.")
    # Überprüfe, ob die Callback-Klasse existiert
    if not hasattr(rpg_training_utils, 'IPyWidgetProgressCallback'):
        raise NameError("Klasse 'IPyWidgetProgressCallback' NICHT in 'rpg_training_utils' gefunden.")


except (NameError, ValueError, AttributeError) as e:
    print(f"FEHLER: Benötigte Module/Variablen/Attribute aus vorherigen Zellen fehlen: {e}")
    print("Bitte stelle sicher, dass die Zellen 1-4 erfolgreich ausgeführt wurden und die Module aktuell sind.")
    raise RuntimeError("Abhängigkeiten für Zelle 5 nicht erfüllt.") from e


# --- Hilfsfunktion für den Trainings-Loop (ANGEPASST für Debug-Widget) ---
# *** HIER DIE ÄNDERUNG: Übergabe des Debug-Widgets ***
def run_training_and_log_report(class_name, total_timesteps, model_dir, log_dir_base, summary_dir, debug_output_widget, load_existing=False):
    """
    Führt das Training für eine Klasse durch und generiert den verständlichen Textbericht.
    Nutzt den IPyWidgetProgressCallback und leitet Debug-Prints in debug_output_widget um.
    """
    print(f"\n{'='*20} Starte Training & Logging für: {class_name} {'='*20}")

    monitor_log_dir = os.path.join(log_dir_base, f"{class_name.lower()}_monitor_logs")
    os.makedirs(monitor_log_dir, exist_ok=True)
    print(f"Monitor Logs werden gespeichert in: {monitor_log_dir}")

    # Zielordner für Trainingsberichte definieren
    training_report_base_dir = os.path.join(summary_dir, "Training Berichte")
    today_str = datetime.date.today().strftime('%Y-%m-%d')
    daily_training_report_dir = os.path.join(training_report_base_dir, today_str)
    os.makedirs(daily_training_report_dir, exist_ok=True)
    print(f"  Tagesordner für Trainingsberichte: {daily_training_report_dir}")

    env_for_training = None
    model = None
    try:
        # 1. Eigene Umgebung erstellen und mit Monitor wrappen
        print("Erstelle NEUE Env-Instanz nur für diesen Trainingslauf...")
        train_env_instance = rpg_env.RPGEnv(
            char_param_definitions=rpg_definitions.CHAR_PARAMS,
            all_buffs_debuffs_names=rpg_definitions.ALL_BUFFS_DEBUFFS_NAMES,
            max_buff_duration=rpg_definitions.MAX_BUFF_DURATION,
            max_stacks=rpg_definitions.MAX_STACKS
        )
        train_env_instance.set_fixed_class(class_name)
        env_for_training = Monitor(train_env_instance, monitor_log_dir, allow_early_resets=True, info_keywords=('is_success',))
        print(f"Neue, gemonitorte Env für Training '{class_name}' erstellt.")

        # 2. Modell erstellen oder laden
        model_path = os.path.join(model_dir, f"ppo_{class_name.lower()}_agent.zip")
        current_steps = 0
        if load_existing and os.path.exists(model_path):
            try:
                print(f"Lade existierendes Modell: {model_path}")
                model = PPO.load(model_path, env=env_for_training)
                current_steps = getattr(model, 'num_timesteps', 0)
                print(f"Modell geladen. Bisherige Schritte: {current_steps}")
            except Exception as e:
                print(f"FEHLER beim Laden von {model_path}: {e}. Erstelle neues Modell.")
                model = None; current_steps = 0
        if model is None:
            print("Erstelle neues PPO-Modell.")
            params = rpg_config.DEFAULT_PPO_PARAMS.copy()
            params['tensorboard_log'] = None
            if 'policy' not in params: params['policy'] = 'MlpPolicy'
            model = PPO(env=env_for_training, **params)
            current_steps = 0

        # 3. Training starten
        remaining_timesteps = total_timesteps - current_steps
        if remaining_timesteps > 0:
            print(f"Starte Training von {current_steps} bis {total_timesteps}...")

            callback_instance = None
            try:
                callback_instance = rpg_training_utils.IPyWidgetProgressCallback(total_timesteps=remaining_timesteps)
            except Exception as e:
                 print(f"WARNUNG: Konnte IPyWidgetProgressCallback nicht erstellen: {e}")

            # *** HIER DIE ÄNDERUNG: model.learn im Kontext des Debug-Widgets ausführen ***
            with debug_output_widget:
                # Alle print() ausgaben innerhalb dieses Blocks (also aus env.step)
                # werden in das debug_output_widget umgeleitet.
                print(f"--- Debug Output für {class_name} ---") # Header für das Widget
                model.learn(
                    total_timesteps=remaining_timesteps,
                    callback=callback_instance,
                    reset_num_timesteps=(not load_existing),
                    progress_bar=False
                )
            # ***************************************************************************

            print("Training abgeschlossen.") # Diese Ausgabe erscheint wieder normal

            # 4. Modell speichern
            latest_path = model_path
            try:
                print(f"Speichere Modell nach: {latest_path}")
                model.save(latest_path)
            except Exception as e:
                 print(f"FEHLER beim Speichern des Modells für {class_name}: {e}")
        else:
             print(f"Ziel-Timesteps ({total_timesteps}) bereits erreicht ({current_steps}). Überspringe Training.")

        # --- 5. Verständlichen Trainings-Bericht generieren ---
        print(f"\nErstelle verständlichen Textbericht für Training von {class_name}...")
        output_file = os.path.join(
            daily_training_report_dir,
            f"trainingsbericht_{class_name.lower()}_verstaendlich.txt"
        )
        config_params_for_log = rpg_config.DEFAULT_PPO_PARAMS

        rpg_training_utils.generate_layperson_log(
            log_dir=monitor_log_dir,
            output_filepath=output_file,
            class_name=class_name,
            total_timesteps=total_timesteps,
            config_params=config_params_for_log
        )

    except Exception as e:
        print(f"\nFEHLER im Trainings-/Logging-Prozess für Klasse {class_name}: {e}")
        # *** HIER DIE ÄNDERUNG: Fehler auch ins Debug-Widget schreiben ***
        with debug_output_widget:
             print(f"\nFEHLER im Trainings-/Logging-Prozess für Klasse {class_name}: {e}")
             traceback.print_exc()
        # ****************************************************************
    finally:
        # 6. Umgebung schließen
        if env_for_training is not None:
            env_for_training.close()
            print(f"Trainingsumgebung für {class_name} geschlossen.")

    print(f"{'='*20} Training & Logging für {class_name} beendet {'='*20}\n")


# --- Haupt-Trainingsschleife ---

# *** HIER DIE ÄNDERUNG: Output Widget erstellen und anzeigen ***
print("Erstelle separates Widget für Debug-Ausgaben...")
debug_output_widget = widgets.Output(layout={
    'border': '1px solid grey',
    'height': '350px', # Höhe anpassen nach Bedarf
    'overflow_y': 'scroll' # Scrollbar hinzufügen
})
display(debug_output_widget) # Widget anzeigen
# ************************************************************


print(f"\n=== Starte Trainingsprozess für Klassen: {rpg_config.CLASSES_TO_TRAIN} ===")
print(f"Gesamt-Timesteps pro Klasse: {rpg_config.TOTAL_TIMESTEPS_PER_CLASS}")
print(f"Modelle laden/speichern in: {rpg_config.MODEL_DIR_BASE}")
print(f"Basis-Log Verzeichnis (Monitor): {rpg_config.LOG_DIR_BASE}")
print(f"Basis-Verzeichnis für Berichte: {rpg_config.SUMMARY_SAVE_DIR}")
print(f"Existierende Modelle laden: {rpg_config.LOAD_EXISTING_MODELS}")
print("-" * 60)

start_time_total = time.time()

try:
    # Aufruf für jede Klasse
    for class_name in rpg_config.CLASSES_TO_TRAIN:
        if class_name not in rpg_definitions.CHAR_PARAMS:
             print(f"WARNUNG: Klasse '{class_name}' nicht in CHAR_PARAMS gefunden. Überspringe.")
             continue

        # *** HIER DIE ÄNDERUNG: Debug-Widget an die Funktion übergeben ***
        run_training_and_log_report(
            class_name=class_name,
            total_timesteps=rpg_config.TOTAL_TIMESTEPS_PER_CLASS,
            model_dir=rpg_config.MODEL_DIR_BASE,
            log_dir_base=rpg_config.LOG_DIR_BASE,
            summary_dir=rpg_config.SUMMARY_SAVE_DIR,
            debug_output_widget=debug_output_widget, # Widget übergeben
            load_existing=rpg_config.LOAD_EXISTING_MODELS
        )
        # ***************************************************************
        print("-" * 60) # Trennlinie zwischen Klassen

except NameError as e:
     print(f"\nFATALER FEHLER: Benötigte Variable/Modul nicht gefunden: {e}")
     print("Stelle sicher, dass Zellen 1-4 ausgeführt wurden und alle Module/Variablen korrekt definiert sind.")
     traceback.print_exc()
except Exception as e:
     print(f"\nFATALER Allgemeiner Fehler im Trainings-Loop: {e}")
     traceback.print_exc()
finally:
    end_time_total = time.time()
    duration_total = end_time_total - start_time_total
    print(f"\n=== Gesamter Trainingsprozess beendet ===")
    print(f"Gesamtdauer: {datetime.timedelta(seconds=duration_total)}")




=== Starte Trainingsprozess für Klassen: ['Krieger', 'Magier', 'Schurke', 'Kleriker'] ===
Gesamt-Timesteps pro Klasse: 50000
Modelle laden/speichern in: /content/drive/MyDrive/Colab Notebooks/models/klassen_agenten_lvl_v2/
Basis-Log Verzeichnis (Monitor): /content/drive/MyDrive/Colab Notebooks/logs/ppo_klassen_lvl_v2/
Basis-Verzeichnis für Berichte: /content/drive/MyDrive/Colab Notebooks/Auswertungsberichte/
Existierende Modelle laden: False
------------------------------------------------------------

==================== Starte Training & Logging für: Krieger ====================
Monitor Logs werden gespeichert in: /content/drive/MyDrive/Colab Notebooks/logs/ppo_klassen_lvl_v2/krieger_monitor_logs
  Tagesordner für Trainingsberichte: /content/drive/MyDrive/Colab Notebooks/Auswertungsberichte/Training Berichte/2025-04-17
Erstelle NEUE Env-Instanz nur für diesen Trainingslauf...
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
Neue, gemonitorte Env für Tra

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Ork Berserker_L1_657 erleidet 36 Schaden.
Ork Berserker_L1_657 benutzt 'Wutanfall' auf Krieger_Hero.
Effekt 'Wutanfall' wird auf Ork Berserker_L1_657 angewendet (Dauer: 3, Mods: {'mod_angriff': 8, 'mod_verteidigung': -5}).
Ork Berserker_L1_657 benutzt 'Wuchtschlag' auf Krieger_Hero.
Krieger_Hero erleidet 54 Schaden.
Krieger_Hero benutzt 'Mächtiger Schlag' auf Ork Berserker_L1_657.
Ork Berserker_L1_657 erleidet 41 Schaden.
Effekt 'Wutanfall' auf Ork Berserker_L1_657 ist abgelaufen.
Krieger_Hero benutzt 'Mächtiger Schlag' auf Ork Berserker_L1_657.
Ork Berserker_L1_657 erleidet 36 Schaden.
Ork Berserker_L1_657 benutzt 'Wuchtschlag' auf Krieger_Hero.
Krieger_Hero erleidet 46 Schaden.
Krieger_Hero benutzt 'Mächtiger Schlag' auf Ork Berserker_L1_657.
Ork Berserker_L1_657 erleidet 36 Schaden.
Ork Berserker_L1_657 wurde besiegt!
Krieger_Hero erhält 25 XP. (Aktuell: 25/100)
Krieger_Hero benutzt 'Mächtiger Schlag' auf Ork Berserk

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Effekt 'Schwächen' wird auf Ork Berserker_L1_550 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Ork Berserker_L1_550 benutzt 'Wuchtschlag' auf Magier_Hero.
Magier_Hero erleidet 58 Schaden.
Effekt 'Wutanfall' auf Ork Berserker_L1_550 ist abgelaufen.
Magier_Hero benutzt 'Feuerball' auf Ork Berserker_L1_550.
Ork Berserker_L1_550 erleidet 46 Schaden.
Effekt 'Schwächen' auf Ork Berserker_L1_550 ist abgelaufen.
Magier_Hero wurde besiegt!
Magier_Hero benutzt 'Feuerball' auf Goblin Schamane_L1_368.
Goblin Schamane_L1_368 erleidet 51 Schaden.
Goblin Schamane_L1_368 benutzt 'Schwächen' auf Magier_Hero.
Effekt 'Schwächen' wird auf Magier_Hero angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Goblin Schamane_L1_368 benutzt 'Blitz' auf Magier_Hero.
Magier_Hero erleidet 9 Schaden.
Magier_Hero benutzt 'Schwächen' auf Goblin Schamane_L1_368.
Effekt 'Schwächen' wird auf Goblin Schamane_

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Effekt 'Vergiften' wird auf Skelett_L1_728 angewendet (Dauer: 4, Mods: {'dot_damage': 5}).
Effekt 'Ausweichen verbessern' auf Schurke_Hero ist abgelaufen.
Skelett_L1_728 erleidet 5 Schaden durch 'Vergiften'.
Schurke_Hero benutzt 'Schneller Stich' auf Skelett_L1_728.
Skelett_L1_728 erleidet 19 Schaden.
Skelett_L1_728 wurde besiegt!
Schurke_Hero erhält 25 XP. (Aktuell: 25/100)
Schurke_Hero benutzt 'Vergiften' auf Skelett Magier_L1_702.
Effekt 'Vergiften' wird auf Skelett Magier_L1_702 angewendet (Dauer: 4, Mods: {'dot_damage': 5}).
Skelett Magier_L1_702 benutzt 'Frostpfeil' auf Schurke_Hero.
Schurke_Hero erleidet 13 Schaden.
Skelett Magier_L1_702 erleidet 5 Schaden durch 'Vergiften'.
Skelett Magier_L1_702 benutzt 'Knochensplitter' auf Schurke_Hero.
Schurke_Hero erleidet 1 Schaden.
Skelett Magier_L1_702 erleidet 5 Schaden durch 'Vergiften'.
Schurke_Hero benutzt 'Schneller Stich' auf Skelett Magier_L1_702.
Skelett Magier_L1

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Effekt 'Schwächen' wird auf Ork_L1_745 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L1_745.
Ork_L1_745 erleidet 28 Schaden.
Ork_L1_745 benutzt 'Wuchtschlag' auf Kleriker_Hero.
Kleriker_Hero erleidet 39 Schaden.
Effekt 'Schwächen' auf Ork_L1_745 ist abgelaufen.
Kleriker_Hero benutzt 'Regeneration' auf Kleriker_Hero.
Effekt 'Regeneration' wird auf Kleriker_Hero angewendet (Dauer: 3, Mods: {'dot_heal': 10}).
Kleriker_Hero heilt 10 HP durch 'Regeneration'.
Kleriker_Hero benutzt 'Leichte Heilung' auf Kleriker_Hero.
Kleriker_Hero wird um 60 HP geheilt.
Kleriker_Hero heilt 10 HP durch 'Regeneration'.
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L1_745.
Ork_L1_745 erleidet 23 Schaden.
Kleriker_Hero heilt 10 HP durch 'Regeneration'.
Effekt 'Regeneration' auf Kleriker_Hero ist abgelaufen.
Ork_L1_745 benutzt 'Wuchtschlag' auf Kleriker_Hero.
Kleriker_He

# 6: TensorBoard starten


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 6: TensorBoard starten (Korrigierter Pfad mit Anführungszeichen)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print("\nStarte TensorBoard...")
# Laden der TensorBoard Notebook Erweiterung (sollte noch geladen sein, schadet aber nicht)
%load_ext tensorboard

# Stelle sicher, dass die Variable LOG_DIR_BASE aus rpg_config existiert
if 'rpg_config' in locals() and hasattr(rpg_config, 'LOG_DIR_BASE'):
    log_directory = rpg_config.LOG_DIR_BASE
    print(f"TensorBoard wird gestartet und überwacht das Verzeichnis: {log_directory}")
    # WICHTIG: Den Pfad in Anführungszeichen setzen!
    %tensorboard --logdir "{log_directory}"
else:
    print("FEHLER: LOG_DIR_BASE nicht in rpg_config gefunden. Kann TensorBoard nicht starten.")
    print("Bitte stelle sicher, dass Zelle 2 (Imports) erfolgreich gelaufen ist.")

# 7: Detaillierte Evaluierung der Agenten

In [ ]:
# -*- coding: utf-8 -*-
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 7: Detaillierte Evaluierung der Agenten
#          (Angepasst V4: Korrekter Import für VBox)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import traceback
import importlib
import os
import sys
# *** HIER DIE ÄNDERUNG: Korrekte Widget-Imports ***
import ipywidgets as widgets
from IPython.display import display # Nur display von hier
from ipywidgets import VBox         # VBox von ipywidgets importieren

# --- Stelle sicher, dass Module aus vorherigen Zellen korrekt geladen sind ---
try:
    print("Prüfe Verfügbarkeit der Module und benötigten Inhalte...")
    # Überprüfe, ob die Module im globalen Scope existieren
    if 'rpg_config' not in globals(): raise NameError("Modul 'rpg_config' nicht global gefunden.")
    if 'rpg_definitions' not in globals(): raise NameError("Modul 'rpg_definitions' nicht global gefunden.")
    if 'rpg_training_utils' not in globals(): raise NameError("Modul 'rpg_training_utils' nicht global gefunden.")
    if 'rpg_env' not in globals(): raise NameError("Modul 'rpg_env' nicht gefunden.") # Hinzugefügt für Env-Check

    # Überprüfe, ob die benötigten Variablen/Funktionen in den Modulen existieren
    if not hasattr(rpg_config, 'CLASSES_TO_TRAIN'): raise NameError("Variable 'CLASSES_TO_TRAIN' nicht in rpg_config gefunden.")
    if not hasattr(rpg_definitions, 'CHAR_PARAMS'): raise NameError("Variable 'CHAR_PARAMS' nicht in rpg_definitions gefunden.")
    if not hasattr(rpg_config, 'MODEL_DIR_BASE'): raise NameError("Variable 'MODEL_DIR_BASE' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'SUMMARY_SAVE_DIR'): raise NameError("Variable 'SUMMARY_SAVE_DIR' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'EVAL_DETERMINISTIC'): raise NameError("Variable 'EVAL_DETERMINISTIC' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'MAX_TURNS'): raise NameError("Variable 'MAX_TURNS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'DEFAULT_PPO_PARAMS'): raise NameError("Variable 'DEFAULT_PPO_PARAMS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_training_utils, 'evaluate_agent_and_summarize'):
         raise NameError("Funktion 'evaluate_agent_and_summarize' nicht in rpg_training_utils gefunden.")
    if not hasattr(rpg_training_utils, 'N_FIGHTS_PER_OPPONENT'):
         raise NameError("Konstante 'N_FIGHTS_PER_OPPONENT' nicht in rpg_training_utils gefunden.")
    if not hasattr(rpg_env, 'RPGEnv'): raise AttributeError("Klasse 'RPGEnv' nicht in Modul 'rpg_env' gefunden.") # Hinzugefügt

    print("Alle benötigten Module und Inhalte scheinen verfügbar zu sein.")

except (NameError, ImportError, AttributeError) as e:
    print(f"FEHLER: Benötigte Module/Variablen/Funktionen nicht gefunden: {e}")
    print("Bitte stelle sicher, dass Zelle 2 erfolgreich und NACH allen '%%writefile'-Befehlen ausgeführt wurde.")
    raise RuntimeError("Abhängigkeiten für Zelle 7 nicht erfüllt.") from e
except Exception as e:
     print(f"FEHLER beim Prüfen der Module: {e}")
     traceback.print_exc()
     raise e


# --- Output Widgets erstellen ---
print("Erstelle separate Widgets für Kampf-Log und Info-Ausgaben...")
combat_log_widget = widgets.Output(layout={
    'border': '1px solid blue',
    'height': '400px', # Höhe anpassen
    'overflow_y': 'scroll',
    'width': '100%'
})
info_output_widget = widgets.Output(layout={
    'border': '1px solid green',
    'height': '200px', # Höhe anpassen
    'overflow_y': 'scroll',
    'width': '100%'
})

# Widgets anzeigen (untereinander)
print("Widgets werden angezeigt (Info oben, Kampf-Log unten):")
display(VBox([info_output_widget, combat_log_widget])) # VBox wird jetzt korrekt gefunden


# --- Evaluierung starten ---
print(f"\n=== Starte detaillierte Evaluierung für Klassen: {rpg_config.CLASSES_TO_TRAIN} ===")
# Parameter-Ausgaben mit Werten aus rpg_config
print(f"Kämpfe pro Gegner: {rpg_training_utils.N_FIGHTS_PER_OPPONENT}")
print(f"Modelle laden aus: {rpg_config.MODEL_DIR_BASE}")
print(f"Zusammenfassungen speichern in (Basisordner): {rpg_config.SUMMARY_SAVE_DIR}")
print(f"Aktionswahl: {'Festgelegt' if rpg_config.EVAL_DETERMINISTIC else 'Zufällig (mit Erkundung)'}")
print("-" * 60)


# --- Schleife für jede Klasse ---
for class_name in rpg_config.CLASSES_TO_TRAIN:
    if class_name not in rpg_definitions.CHAR_PARAMS:
         print(f"\nWARNUNG: Klasse '{class_name}' nicht in CHAR_PARAMS gefunden. Überspringe Evaluierung.")
         continue

    try:
        # Widgets an die Funktion übergeben
        rpg_training_utils.evaluate_agent_and_summarize(
            class_name=class_name,
            model_dir_base=rpg_config.MODEL_DIR_BASE,
            summary_dir=rpg_config.SUMMARY_SAVE_DIR,
            deterministic=rpg_config.EVAL_DETERMINISTIC,
            max_turns_per_episode=rpg_config.MAX_TURNS,
            training_params=rpg_config.DEFAULT_PPO_PARAMS,
            output_format='txt_single',
            combat_log_widget=combat_log_widget, # Kampf-Log Widget übergeben
            info_output_widget=info_output_widget   # Info Widget übergeben
        )

    except Exception as e:
        print(f"\nFEHLER bei der Evaluierung von Klasse {class_name}: {e}")
        # Fehler auch ins Info-Widget schreiben, falls möglich
        if info_output_widget:
             with info_output_widget:
                 print(f"\nFEHLER bei der Evaluierung von Klasse {class_name}: {e}")
                 traceback.print_exc()
        else:
             traceback.print_exc()


    print("-" * 60) # Trennlinie

print(f"\n=== Detaillierte Evaluierung für alle Klassen abgeschlossen (oder versucht) ===")
print(f"Überprüfe die .txt/.csv/.html-Dateien im Ordner (bzw. dessen Tages-Unterordnern): {rpg_config.SUMMARY_SAVE_DIR}")



Prüfe Verfügbarkeit der Module und benötigten Inhalte...
Alle benötigten Module und Inhalte scheinen verfügbar zu sein.
Erstelle separate Widgets für Kampf-Log und Info-Ausgaben...
Widgets werden angezeigt (Info oben, Kampf-Log unten):


/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_layout.py:84: DeprecationWarning: Layout properties overflow_x and overflow_y have been deprecated and will be dropped in a future release. Please use the overflow shorthand property instead
  warnings.warn("Layout properties overflow_x and overflow_y have been deprecated and will be dropped in a future release. Please use the overflow shorthand property instead", DeprecationWarning)



=== Starte detaillierte Evaluierung für Klassen: ['Krieger', 'Magier', 'Schurke', 'Kleriker'] ===
Kämpfe pro Gegner: 50
Modelle laden aus: /content/drive/MyDrive/Colab Notebooks/models/klassen_agenten_lvl_v2/
Zusammenfassungen speichern in (Basisordner): /content/drive/MyDrive/Colab Notebooks/Auswertungsberichte/
Aktionswahl: Festgelegt
------------------------------------------------------------
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
------------------------------------------------------------
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
------------------------------------------------------------
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
------------------------------------------------------------
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
------------------------------------------------------------

=== Detaillierte Evaluierung für alle Klas

In [ ]:
# -*- coding: utf-8 -*-
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 7: Detaillierte Evaluierung der Agenten
#          (Angepasst V2: Vereinfachter Import-Check, nutzt global geladene Module)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import traceback
import importlib # Bleibt für ggf. manuelle Reloads nützlich
import os
import sys # Für sys.modules Check

# --- Stelle sicher, dass Module aus vorherigen Zellen korrekt geladen sind ---
# Annahme: Zelle 2 hat die Module importiert und ggf. neu geladen!
try:
    print("Prüfe Verfügbarkeit der Module und benötigten Inhalte...")
    # Überprüfe, ob die Module im globalen Scope existieren
    if 'rpg_config' not in globals(): raise NameError("Modul 'rpg_config' nicht global gefunden.")
    if 'rpg_definitions' not in globals(): raise NameError("Modul 'rpg_definitions' nicht global gefunden.")
    if 'rpg_training_utils' not in globals(): raise NameError("Modul 'rpg_training_utils' nicht global gefunden.")

    # Überprüfe, ob die benötigten Variablen/Funktionen in den Modulen existieren
    if not hasattr(rpg_config, 'CLASSES_TO_TRAIN'): raise NameError("Variable 'CLASSES_TO_TRAIN' nicht in rpg_config gefunden.")
    if not hasattr(rpg_definitions, 'CHAR_PARAMS'): raise NameError("Variable 'CHAR_PARAMS' nicht in rpg_definitions gefunden.")
    if not hasattr(rpg_config, 'MODEL_DIR_BASE'): raise NameError("Variable 'MODEL_DIR_BASE' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'SUMMARY_SAVE_DIR'): raise NameError("Variable 'SUMMARY_SAVE_DIR' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'EVAL_DETERMINISTIC'): raise NameError("Variable 'EVAL_DETERMINISTIC' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'MAX_TURNS'): raise NameError("Variable 'MAX_TURNS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'DEFAULT_PPO_PARAMS'): raise NameError("Variable 'DEFAULT_PPO_PARAMS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_training_utils, 'evaluate_agent_and_summarize'):
         raise NameError("Funktion 'evaluate_agent_and_summarize' nicht in rpg_training_utils gefunden.")
    # Überprüfe die Konstante erneut
    if not hasattr(rpg_training_utils, 'N_FIGHTS_PER_OPPONENT'):
         raise NameError("Konstante 'N_FIGHTS_PER_OPPONENT' nicht in rpg_training_utils gefunden.")

    print("Alle benötigten Module und Inhalte scheinen verfügbar zu sein.")

except (NameError, ImportError, AttributeError) as e:
    print(f"FEHLER: Benötigte Module/Variablen/Funktionen nicht gefunden: {e}")
    print("Bitte stelle sicher, dass Zelle 2 erfolgreich und NACH allen '%%writefile'-Befehlen ausgeführt wurde.")
    raise RuntimeError("Abhängigkeiten für Zelle 7 nicht erfüllt.") from e
except Exception as e:
     print(f"FEHLER beim Prüfen der Module: {e}")
     traceback.print_exc()
     raise e


# --- Evaluierung starten ---
print(f"\n=== Starte detaillierte Evaluierung für Klassen: {rpg_config.CLASSES_TO_TRAIN} ===")
# Parameter-Ausgaben mit Werten aus rpg_config
print(f"Kämpfe pro Gegner: {rpg_training_utils.N_FIGHTS_PER_OPPONENT}") # Zugriff auf die Konstante
print(f"Modelle laden aus: {rpg_config.MODEL_DIR_BASE}")
print(f"Zusammenfassungen speichern in (Basisordner): {rpg_config.SUMMARY_SAVE_DIR}")
print(f"Aktionswahl: {'Festgelegt' if rpg_config.EVAL_DETERMINISTIC else 'Zufällig (mit Erkundung)'}")
print("-" * 60)


# --- Schleife für jede Klasse ---
for class_name in rpg_config.CLASSES_TO_TRAIN:
    if class_name not in rpg_definitions.CHAR_PARAMS:
         print(f"\nWARNUNG: Klasse '{class_name}' nicht in CHAR_PARAMS gefunden. Überspringe Evaluierung.")
         continue

    try:
        # --- Korrekter Aufruf für V19+ von evaluate_agent_and_summarize ---
        rpg_training_utils.evaluate_agent_and_summarize(
            class_name=class_name,
            model_dir_base=rpg_config.MODEL_DIR_BASE,
            summary_dir=rpg_config.SUMMARY_SAVE_DIR, # Basisordner übergeben
            deterministic=rpg_config.EVAL_DETERMINISTIC,
            max_turns_per_episode=rpg_config.MAX_TURNS,
            training_params=rpg_config.DEFAULT_PPO_PARAMS, # Optional übergeben
            output_format='txt_single' # Hier Ausgabeformat wählen
        )
        # --- Ende Funktionsaufruf ---

    except Exception as e:
        print(f"\nFEHLER bei der Evaluierung von Klasse {class_name}: {e}")
        traceback.print_exc()

    print("-" * 60) # Trennlinie

print(f"\n=== Detaillierte Evaluierung für alle Klassen abgeschlossen (oder versucht) ===")
print(f"Überprüfe die .txt/.csv/.html-Dateien im Ordner (bzw. dessen Tages-Unterordnern): {rpg_config.SUMMARY_SAVE_DIR}")

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Kleriker_Hero benutzt 'Schwächen' auf Ork_L2_597.
Effekt 'Schwächen' wird auf Ork_L2_597 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Segnen' auf Kleriker_Hero.
Effekt 'Segnen' wird auf Kleriker_Hero angewendet (Dauer: 4, Mods: {'mod_verteidigung': 5, 'mod_magiekraft': 3}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L2_597.
Ork_L2_597 erleidet 31 Schaden.
Ork_L2_597 benutzt 'Wuchtschlag' auf Kleriker_Hero.
Kleriker_Hero erleidet 34 Schaden.
Effekt 'Schwächen' auf Ork_L2_597 ist abgelaufen.
Kleriker_Hero benutzt 'Leichte Heilung' auf Kleriker_Hero.
Kleriker_Hero wird um 63 HP geheilt.
Effekt 'Segnen' auf Kleriker_Hero ist abgelaufen.
Kleriker_Hero benutzt 'Schwächen' auf Ork_L2_597.
Effekt 'Schwächen' wird auf Ork_L2_597 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L2_597.
Ork_L2_597 erlei

# 8: Beispielhafte UI-Anzeige (Optional)

In [ ]:
print("\nOptional: Zeige ein Beispiel der HTML UI...")

if rpg_environment_instance:
    try:
        # Setze die Umgebung für eine Beispielklasse zurück
        example_class = random.choice(rpg_config.CLASSES_TO_TRAIN)
        rpg_environment_instance.set_fixed_class(example_class)
        obs, info = rpg_environment_instance.reset()

        # Erzeuge ein paar Beispiel-Logeinträge
        example_logs = [
            f"Kampf beginnt! {rpg_environment_instance.hero.name} vs {rpg_environment_instance.enemy.name}",
            f"{rpg_environment_instance.hero.name} setzt Feuerball ein.",
            f"{rpg_environment_instance.enemy.name} erleidet 35 Schaden.",
            f"{rpg_environment_instance.enemy.name} wendet Vergiften auf {rpg_environment_instance.hero.name} an.",
            f"{rpg_environment_instance.hero.name} erleidet 5 Schaden durch Gift.",
            f"{rpg_environment_instance.hero.name} wird um 30 geheilt.",
            f"{rpg_environment_instance.enemy.name} wurde besiegt!"
        ]

        # Generiere und zeige die UI
        html_output = rpg_ui.generate_html_ui(rpg_environment_instance, example_logs)
        print(f"Zeige UI für Beispielszene (Klasse: {example_class}):")
        display(html_output)

    except Exception as e:
        print(f"FEHLER beim Erzeugen der Beispiel-UI: {e}")
        traceback.print_exc()
else:
    print("Keine RPGEnv Instanz für UI-Beispiel verfügbar.")

# 9: Hilfsprogramme (Summary Generatoren)


In [12]:
    # -*- coding: utf-8 -*-
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    # ZELLE 9.1: Hilfsskript zum Generieren der Entscheidungs-Zusammenfassung (TXT)
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

    import os
    import datetime

    # *** HIER DIE ÄNDERUNG: Standard-Dateiname auf .txt geändert ***
    def generate_decision_summary_txt(filename="ENTSCHEIDUNGEN_AKTUELL.txt"):
        """
        Generiert eine Text-Datei mit der detaillierten Zusammenfassung aller
        Projektentscheidungen.
        """
        print(f"Generiere detaillierte Entscheidungszusammenfassung nach '{filename}'...")

        # Hole den aktuellen Zeitstempel
        now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Der gesamte Text der Zusammenfassung, Markdown entfernt/vereinfacht
        summary_text = f"""Projektentscheidungen & Status: Python RPG (Stand: {now})
    =========================================================

    Dieses Dokument fasst die grundlegenden und aktuellen Design- und Implementierungsentscheidungen für das selbstlaufende Python-RPG-Projekt zusammen.

    1. Grundlegende Architektur & Umgebung
    ---------------------------------------
    - Ziel: Entwicklung eines selbstlaufenden RPGs in Python, primär zur Beobachtung von KI-Verhalten (Skript-basiert und Reinforcement Learning).
    - Entwicklungsumgebung: Google Colab mit einem zentralen Jupyter Notebook (RPG.ipynb).
    - Modularisierung: Code ist in separate .py-Dateien ausgelagert (rpg_config.py, rpg_definitions.py, rpg_game_logic.py, rpg_env.py, rpg_ui.py, rpg_training_utils.py) zur besseren Organisation und Wiederverwendbarkeit. Updates erfolgen oft via %%writefile.
    - Persistenz: Projektstand wird primär durch das Notebook und die .py-Dateien gesichert. RL-Modelle werden als .zip-Dateien gespeichert. Wichtige Entscheidungen werden manuell oder per Skript in .md/.txt-Dateien festgehalten.

    2. Spiel-Logik (rpg_game_logic.py)
    ----------------------------------
    - Klassenbasiert: Implementierung mit Klassen für Character, Skill und Environment.
    - Charakterattribute: Definierte Basisattribute (STR, GES, KON, INT, WEI) und abgeleitete Kampfstatistiken (Angriff, Verteidigung, Magiekraft etc.). Ressourcen: HP, Mana, Stamina, Energy mit passiver Regeneration.
    - Skills: Skills haben Kosten (Ressourcentyp variabel), Abklingzeiten, Zieltypen, Effekttypen (Schaden, Heilung, Buff, Debuff), Stärke, optional Elemente und Dauer.
    - Effektsystem: Unterstützung für Buffs, Debuffs, Schaden-über-Zeit (DoT) und Heilung-über-Zeit (HoT) über ein active_effects-Dictionary in der Character-Klasse. Berücksichtigt Dauer und (optional) Stacking.
    - XP & Leveling: Helden sammeln XP durch das Besiegen von Gegnern und steigen im Level auf, was Attribute erhöht und Ressourcen wiederherstellt.
    - Kampfablauf (Basis): Rundenbasiert. Charaktere agieren basierend auf Initiative (aktuell GES). Start-of-Turn-Effekte (Cooldowns, DoT/HoT, Regeneration) werden angewendet.

    3. Künstliche Intelligenz (KI)
    -----------------------------
    - Hybrid-Ansatz:
        - Gegner-KI: Skriptbasiert (_scripted_enemy_ai in rpg_env.py). Nutzt einfache Heuristiken (Heilung bei niedriger HP, Debuffs anwenden, Buffs anwenden, stärksten Schadensskill nutzen, sonst Basis-Angriff).
        - Helden-KI: Reinforcement Learning (RL) mit stable-baselines3 (Algorithmus: PPO).
    - Agenten-Strategie: Ein separater PPO-Agent wird für jede Heldenklasse trainiert.

    4. RL-Umgebung (rpg_env.py)
    --------------------------
    - Basis: Implementiert als gymnasium.Env-Subklasse.
    - Action Space: spaces.Discrete. Aktionen umfassen alle definierten Skills sowie dedizierte Aktionen für "Passen" und "Basis-Angriff".
    - Observation Space: spaces.Box (kontinuierlich, normalisiert auf 0-1). Beinhaltet:
        - Normalisierte Ressourcen (HP, Mana, etc.) von Held und Gegner.
        - Normalisierte Skill-Cooldowns des Helden.
        - Status von Buffs/Debuffs (Dauer, Stacks) auf Held und Gegner.
        - Normalisiertes Level und XP-Fortschritt des Helden.
    - reset()-Methode: Initialisiert einen Kampf. Wählt Heldenklasse und Gegnertyp aus (kann für Training/Evaluierung fixiert werden via set_fixed_class/set_fixed_opponent). Setzt Level basierend auf einer leichten Varianz.
    - step()-Methode: Verarbeitet die Aktion des Helden-Agenten, führt die Heldenaktion aus, lässt die Gegner-KI agieren, wendet Effekte an, prüft auf Kampfende (Tod eines Charakters oder Erreichen von MAX_TURNS), berechnet die Belohnung und gibt die nächste Observation zurück. Enthält is_success-Info für Monitoring.
    - Reward Shaping: Umfasst Belohnungen/Strafen für:
        - Sieg/Niederlage/Timeout.
        - Verursachten Schaden (relativ zur max HP des Gegners).
        - Rest-HP bei Sieg.
        - Level-Ups.
        - Erfolgreiche Skill-Nutzung.
        - Anwenden spezifischer nützlicher Buffs/Debuffs.
        - Malus für Passen, Basis-Angriff (Held), ungültige Aktionen, lange Kämpfe, Skill-Fehlversuche.
    - Debugging: Enthält aktuell print-Anweisungen in step() zur Diagnose von Endlosschleifen (sollten nach Behebung entfernt werden).

    5. Benutzeroberfläche (UI) (rpg_ui.py)
    --------------------------------------
    - Technologie: Dynamisch generiertes HTML mit CSS, angezeigt in Colab via display(HTML(...)).
    - Zweck: Primär zur visuellen Beobachtung des Spielzustands und Kampfablaufs.
    - Inhalt: Zeigt Charakterboxen mit HP, Ressourcen, Level/XP, aktiven Effekten und Cooldowns an. Enthält ein farblich hervorgehobenes Kampf-Log.

    6. Training (Zelle 5, rpg_training_utils.py)
    ---------------------------------------------
    - Workflow: Haupt-Trainingsschleife in Zelle 5 iteriert über CLASSES_TO_TRAIN (aus rpg_config.py).
    - Umgebung: Für jeden Trainingslauf wird eine eigene RPGEnv-Instanz erstellt und mit Monitor gewrapped, um Statistiken (Belohnung, Länge, Erfolg) in monitor.csv zu loggen.
    - Modell: PPO-Modell wird pro Klasse erstellt oder geladen (LOAD_EXISTING_MODELS Flag).
    - Fortschritt: Visuelle Fortschrittsanzeige via IPyWidgetProgressCallback (Balken, Zeitangaben).
    - Berichterstellung: Nach jedem Klassentraining wird automatisch ein verständlicher Textbericht (trainingsbericht_*.txt) mittels generate_layperson_log erstellt, der den Trainingsfortschritt zusammenfasst. Berichte werden in täglichen Unterordnern gespeichert.
    - Debugging: Debug-Ausgaben aus rpg_env.py werden während des Trainings in ein separates Output-Widget umgeleitet (konfiguriert in Zelle 5).

    7. Evaluierung (Zelle 7, rpg_training_utils.py)
    ----------------------------------------------
    - Workflow: Haupt-Evaluierungsschleife in Zelle 7 iteriert über CLASSES_TO_TRAIN.
    - Funktion: evaluate_agent_and_summarize führt die Evaluierung pro Klasse durch.
    - Durchführung: Lässt den trainierten Agenten eine feste Anzahl von Kämpfen (N_FIGHTS_PER_OPPONENT) gegen definierte Gegnertypen spielen. Aktionswahl deterministisch oder stochastisch (EVAL_DETERMINISTIC).
    - Datenerfassung: Sammelt detaillierte Statistiken pro Kampf und aggregiert sie (Sieg/Niederlage/Timeout-Raten, Ø Belohnung, Ø Dauer, Ø Rest-Ressourcen, Ø Schaden/Heilung, Skill-Nutzung, Passen/Basis-Angriff-Details). Erfasst Statistiken auch pro Gegnertyp.
    - Logging: Schreibt ein detailliertes Schritt-Log (evaluation_steps_*.csv) mit dem Zustand und den Aktionen in jeder Runde jedes Kampfes.
    - Berichterstellung: Generiert automatisch detaillierte Zusammenfassungen als .txt-Datei (optional auch .csv für Vergleich, .html für Webansicht). Berichte werden in täglichen Unterordnern gespeichert.
    - Ausgabe-Management: Leitet während der Ausführung die allgemeinen Statusmeldungen und das detaillierte Kampf-Log in separate ipywidgets.Output-Widgets um (konfiguriert in Zelle 7), um die Übersichtlichkeit zu verbessern.

    8. Konfiguration (rpg_config.py, rpg_definitions.py)
    ----------------------------------------------------
    - rpg_config.py: Enthält globale Konfigurationen wie Verzeichnispfade, zu trainierende Klassen, Trainings-Timesteps, PPO-Hyperparameter, Evaluierungsparameter (MAX_TURNS, EVAL_DETERMINISTIC).
    - rpg_definitions.py: Definiert die "Datenbank" des Spiels: Basisattribute der Charakterklassen (CHAR_PARAMS), Skill-Parameter (SKILL_PARAMS), Zuordnung von Skills zu Klassen (ALL_SKILLS_MAP), Primärskills (PRIMARY_SKILLS), Hauptressourcen (CLASS_MAIN_RESOURCE), Liste aller Buff-/Debuff-Namen.

    9. Aktuelle Herausforderungen & Nächste Schritte
    ------------------------------------------------
    - Balance: Größte aktuelle Herausforderung. Orks sind zu stark, Skelettmagier zu schwach, Krieger oft dominant. Mehrere Balance-Anpassungen waren bisher nicht ausreichend. Ein radikales Rebalancing ist erforderlich.
    - Debugging: Die Ursache für potenziell unendliche Kämpfe (Endlosschleife in Zelle 5) wird aktuell mittels Debug-Ausgaben in rpg_env.py untersucht.
    - Analyse: Nutzung der generierten Evaluierungsberichte und Schritt-Logs zur genaueren Analyse des Agentenverhaltens (z.B. Kleriker-Fehlversuche, spezifische Kampfverläufe).
    """

        try:
            # Schreibe den Text in die Text-Datei
            with open(filename, "w", encoding="utf-8") as f:
                f.write(summary_text)
            print(f"Entscheidungszusammenfassung erfolgreich in '{filename}' geschrieben.")
        except IOError as e:
            print(f"FEHLER beim Schreiben der Datei '{filename}': {e}")
        except Exception as e:
            print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")

    # --- Skript ausführen ---
    # Erzeugt die Text-Datei, wenn diese Zelle ausgeführt wird
    generate_decision_summary_txt()

    # Optional: Zweite Funktion für eine kürzere Projektzusammenfassung (falls benötigt)
    # def generate_project_summary_txt(filename="Projekt_Zusammenfassung.txt"):
    #     # ... (Ähnliche Logik mit kürzerem Text) ...
    #     pass
    # generate_project_summary_txt()


Generiere detaillierte Entscheidungszusammenfassung nach 'ENTSCHEIDUNGEN_AKTUELL.txt'...
Entscheidungszusammenfassung erfolgreich in 'ENTSCHEIDUNGEN_AKTUELL.txt' geschrieben.


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 9.1: Generator für ENTSCHEIDUNGEN.txt
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import datetime
import os
import traceback

print("Aktualisiere ENTSCHEIDUNGEN.txt...")

# Aktuelles Datum und Uhrzeit holen
try:
    now = datetime.datetime.now()
    # Verwende ein Format, das für Dateinamen und Anzeige gut ist
    timestamp_str = now.strftime("%Y-%m-%d %H:%M:%S")
except Exception as e:
    print(f"Fehler beim Holen des Zeitstempels: {e}")
    timestamp_str = "UNBEKANNT"

# --- Der Inhalt der Datei ---
# HINWEIS: Du kannst diesen Textblock jederzeit anpassen, um den aktuellen Stand widerzuspiegeln!
# Es ist sinnvoll, hier die wichtigsten Punkte aus der obigen Markdown-Zusammenfassung einzufügen.
content = f"""
# Projektentscheidungen & Status: Python RPG (Stand: {timestamp_str})

Dieses Dokument fasst wichtige grundlegende und aktuelle Design- und Implementierungsentscheidungen sowie den aktuellen Status des Projekts zusammen.

## 1. Grundlegende Projektdefinition

* **Ziel:** Selbstlaufendes RPG in Python zur Beobachtung von KI-Verhalten.
* **Umgebung:** Python, Google Colab, `.py`-Module, Haupt-Notebook (`RPG.ipynb`).
* **UI:** Dynamisches HTML in Colab (`rpg_ui.py`) zur Beobachtung.
* **KI-Strategie:** Skript-KI für Gegner (Regel-basiert V2.2), RL (PPO) für Helden.
* **Kampfmodus:** Helden treten nur gegen dedizierte Gegnertypen an.

## 2. Wichtige Entwicklungen / Kern-Entscheidungen (Auswahl)

* Basis-System, RL-Agenten, Spielmechanik, Reward Shaping implementiert.
* Diverse Debugging-Phasen durchlaufen (Imports, Reloads, Logik).
* Gegner-Strategie angepasst (kein Held vs. Held).
* Gegner-Pool erweitert (Goblin Varianten, Ork Berserker, Skelett Magier).
* Gegner-KI verbessert (Buff/Debuff/Heal/BA-Logik).
* Gegner-Aktions-Tracking und Basis-Angriff implementiert.
* Mehrere Balance-Anpassungen durchgeführt (V14, V15, V16) - bisher wenig erfolgreich.
* Detaillierte Evaluierungsberichte und Schritt-Logs implementiert.

## 3. Aktueller Stand & Analyse (Basierend auf Eval mit Defs V15 / Env V22)

* **KI Funktionalität:** Gegner-KI (V2.2) und Helden-KI (PPO) laufen technisch.
* **Helden-Performance (Kurz):** Krieger (dominant), Kleriker (gut, aber Fehler/vs Ork), Schurke/Magier (mittel, Probleme vs Ork).
* **Gegner-Balance:** Orks (Basis/Berserker) zu stark. Skelett Magier zu schwach. Goblins OK.
* **Gesamtbalance:** Schieflage durch Orks und Krieger-Dominanz.
* **Offene Punkte:** Ork-Übermacht, Skelett-Magier-Schwäche, Krieger-Dominanz, Kleriker-Fehlversuche analysieren, N/A-Aktionen im Log prüfen.

## 4. Nächste Schritte (Fokus)

* **Priorität:** Radikales Rebalancing (Orks, Skelett Magier, Krieger) in `rpg_definitions.py`.
* **Analyse:** Nutzung der Schritt-Logs (`evaluation_steps_*.csv`) zur Detailanalyse (Kleriker-Fehler, Ork-Kämpfe).
* **Code-Bereinigung:** Redundante Funktionsdefinition in Zelle 5 entfernen. Sicherstellen, dass `N_FIGHTS_PER_OPPONENT` korrekt in `rpg_training_utils.py` definiert ist (V24 empfohlen).
* **Optional:** Berichtsformate aus Evaluierung anpassen (Multi-Format V18/V19 oder bei V23 bleiben).

## 5. Offene Punkte (Generell)

* Langfristige Erweiterungen (Items, Quests etc.) zurückgestellt.
"""

# Schreibe den Inhalt in die Datei
# Stelle sicher, dass der Pfad korrekt ist (aus Zelle 1/2)
try:
    # Prüfe, ob project_path existiert und ein Verzeichnis ist
    if 'project_path' in locals() and isinstance(project_path, str) and os.path.isdir(project_path):
        pass # Alles gut
    elif 'project_path' in globals() and isinstance(globals()['project_path'], str) and os.path.isdir(globals()['project_path']):
         project_path = globals()['project_path'] # Hole aus globalem Scope
    else:
        # Versuche, den Pfad aus rpg_config zu holen, falls Zelle 2 nicht lief, aber Config geladen wurde
        if 'rpg_config' in locals() and hasattr(rpg_config, 'BASE_PROJECT_PATH'):
            project_path = rpg_config.BASE_PROJECT_PATH
            print(f"WARNUNG: 'project_path' nicht gefunden, verwende BASE_PROJECT_PATH aus rpg_config: {project_path}")
            if not os.path.isdir(project_path):
                 print(f"WARNUNG: Fallback-Pfad '{project_path}' ist auch ungültig. Verwende aktuelles Verzeichnis.")
                 project_path = os.getcwd()
        else:
            print("WARNUNG: Variable 'project_path' nicht gefunden oder ungültig. Verwende aktuelles Verzeichnis.")
            project_path = os.getcwd()
except NameError:
     print("WARNUNG: Variable 'project_path' nicht definiert. Verwende aktuelles Verzeichnis.")
     project_path = os.getcwd()

# Definiere den Dateinamen als .txt
file_name = "ENTSCHEIDUNGEN.txt"
file_path = os.path.join(project_path, file_name)
print(f"Versuche in Datei zu schreiben: {file_path}")

try:
    # Stelle sicher, dass das Zielverzeichnis existiert
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        # Entferne führende/trailing Leerzeichen/Zeilenumbrüche vom content
        f.write(content.strip() + "\n") # Füge einen einzelnen Zeilenumbruch am Ende hinzu
    print(f"Datei '{file_name}' erfolgreich unter '{file_path}' geschrieben/aktualisiert.")
except Exception as e:
    print(f"FEHLER beim Schreiben der Datei '{file_name}': {e}")
    print(f"Versuchter Pfad: {file_path}")
    traceback.print_exc()



Aktualisiere ENTSCHEIDUNGEN.txt...
Versuche in Datei zu schreiben: /content/drive/MyDrive/Colab Notebooks/ENTSCHEIDUNGEN.txt
Datei 'ENTSCHEIDUNGEN.txt' erfolgreich unter '/content/drive/MyDrive/Colab Notebooks/ENTSCHEIDUNGEN.txt' geschrieben/aktualisiert.


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 9.3: Backup von .ipynb und .py Dateien als .txt
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import os
import shutil
import datetime
import traceback

print("Starte Backup-Prozess für .ipynb und .py Dateien als .txt...")

# --- Konfiguration ---
# Stelle sicher, dass der Projektpfad korrekt ist (aus Zelle 1/2)
if 'project_path' not in locals() or not os.path.isdir(project_path):
     print("WARNUNG: 'project_path' nicht korrekt gesetzt oder Verzeichnis existiert nicht.")
     project_path = os.getcwd()
     print(f"Verwende aktuelles Verzeichnis als Fallback: {project_path}")

# Name des Notebooks (ggf. anpassen)
notebook_filename = "RPG.ipynb" # Passe dies an, falls dein Notebook anders heißt

# Liste der Python-Module (Namen ohne .py)
# Du kannst diese Liste manuell pflegen oder versuchen, sie dynamisch zu bekommen
# Hier verwenden wir die Liste, die wir kennen:
module_names = [
    'rpg_config',
    'rpg_definitions',
    'rpg_game_logic',
    'rpg_ui',
    'rpg_env',
    'rpg_training_utils'
]

# Verzeichnis für die Text-Backups erstellen
backup_dir_name = "backup_txt"
backup_dir_path = os.path.join(project_path, backup_dir_name)
try:
    os.makedirs(backup_dir_path, exist_ok=True)
    print(f"Backup-Verzeichnis sichergestellt: {backup_dir_path}")
except Exception as e:
    print(f"FEHLER beim Erstellen des Backup-Verzeichnisses: {e}")
    # Hier vielleicht abbrechen, wenn das Verzeichnis nicht erstellt werden kann?
    # raise e # würde die Zelle stoppen

# --- Dateien zum Sichern zusammenstellen ---
files_to_backup = []
# Notebook hinzufügen
notebook_path = os.path.join(project_path, notebook_filename)
if os.path.exists(notebook_path):
    files_to_backup.append(notebook_filename)
else:
    print(f"WARNUNG: Notebook-Datei '{notebook_filename}' nicht im Projektpfad gefunden.")

# Python-Dateien hinzufügen
for module_name in module_names:
    py_filename = module_name + ".py"
    py_path = os.path.join(project_path, py_filename)
    if os.path.exists(py_path):
        files_to_backup.append(py_filename)
    else:
        print(f"WARNUNG: Python-Datei '{py_filename}' nicht im Projektpfad gefunden.")

print(f"\nFolgende Dateien werden als .txt gesichert (falls vorhanden): {files_to_backup}")

# --- Backup-Vorgang ---
backup_count = 0
error_count = 0
for filename in files_to_backup:
    source_path = os.path.join(project_path, filename)
    # Backup-Dateiname: original.py -> original.py.txt oder original.ipynb -> original.ipynb.txt
    backup_filename = filename + ".txt"
    dest_path = os.path.join(backup_dir_path, backup_filename)

    try:
        print(f"  Sichere '{filename}' nach '{backup_filename}'...")
        # Kopiere den Inhalt Zeile für Zeile (sicherer für verschiedene Encodings)
        with open(source_path, 'r', encoding='utf-8', errors='ignore') as f_source, \
             open(dest_path, 'w', encoding='utf-8') as f_dest:
            for line in f_source:
                f_dest.write(line)
        backup_count += 1
        print(f"    -> Erfolgreich.")
    except FileNotFoundError:
        print(f"    -> FEHLER: Quelldatei '{filename}' nicht gefunden (sollte nicht passieren).")
        error_count += 1
    except Exception as e:
        print(f"    -> FEHLER beim Sichern von '{filename}': {e}")
        traceback.print_exc()
        error_count += 1

print(f"\nBackup-Prozess abgeschlossen.")
print(f"- Erfolgreich gesichert: {backup_count} Datei(en)")
if error_count > 0:
    print(f"- Fehler aufgetreten: {error_count} Datei(en)")
print(f"- Backups gespeichert in: {backup_dir_path}")

Starte Backup-Prozess für .ipynb und .py Dateien als .txt...
Backup-Verzeichnis sichergestellt: /content/drive/MyDrive/Colab Notebooks/backup_txt

Folgende Dateien werden als .txt gesichert (falls vorhanden): ['RPG.ipynb', 'rpg_config.py', 'rpg_definitions.py', 'rpg_game_logic.py', 'rpg_ui.py', 'rpg_env.py', 'rpg_training_utils.py']
  Sichere 'RPG.ipynb' nach 'RPG.ipynb.txt'...
    -> Erfolgreich.
  Sichere 'rpg_config.py' nach 'rpg_config.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_definitions.py' nach 'rpg_definitions.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_game_logic.py' nach 'rpg_game_logic.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_ui.py' nach 'rpg_ui.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_env.py' nach 'rpg_env.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_training_utils.py' nach 'rpg_training_utils.py.txt'...
    -> Erfolgreich.

Backup-Prozess abgeschlossen.
- Erfolgreich gesichert: 7 Datei(en)
- Backups gespeichert in: /content/drive/MyDrive/Colab Notebooks/ba

## 10: Manueller Test für XP & Level Up

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE (z.B. 10): Manueller Test für XP & Level Up (inkl. manuellem LvlUp-Check)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import importlib
import time
import traceback # Für detailliertere Fehlermeldungen
from IPython.display import display, clear_output
import random

# --- Sicherstellen, dass die UI Funktion existiert ---
# (Normalerweise durch Zelle 2 importiert, aber zur Sicherheit hier prüfen)
if 'rpg_ui' not in locals() or not hasattr(rpg_ui, 'generate_html_ui'):
     print("FEHLER: UI-Funktion 'generate_html_ui' nicht gefunden. Bitte Zelle 2 ausführen.")
     # Hier besser abbrechen
     raise NameError("UI-Funktion nicht verfügbar.")


print("--- Starte Manuellen Testlauf ---")

# --- 1. Module neu laden (WICHTIG!) ---
# Lädt die neuesten Versionen der .py Dateien
try:
    # Stelle sicher, dass die Variablen aus vorherigen Zellen existieren
    if 'rpg_definitions' not in locals(): raise NameError("Modul 'rpg_definitions' nicht importiert.")
    if 'rpg_game_logic' not in locals(): raise NameError("Modul 'rpg_game_logic' nicht importiert.")
    if 'rpg_env' not in locals(): raise NameError("Modul 'rpg_env' nicht importiert.")
    if 'rpg_ui' not in locals(): raise NameError("Modul 'rpg_ui' nicht importiert.")

    importlib.reload(rpg_definitions)
    importlib.reload(rpg_game_logic)
    importlib.reload(rpg_env)
    importlib.reload(rpg_ui)
    print("Module rpg_definitions, rpg_game_logic, rpg_env, rpg_ui neu geladen.")
except NameError as e:
    print(f"FEHLER: Konnte Module nicht neu laden, da sie nicht importiert wurden. Stelle sicher, dass Zelle 2/3 gelaufen sind. Fehler: {e}")
    raise e
except Exception as e:
    print(f"FEHLER beim Neuladen der Module: {e}")
    traceback.print_exc()
    raise e

# --- 2. Umgebung holen oder neu erstellen ---
env = None # Sicherstellen, dass env definiert ist
# Versuche, die bestehende Instanz aus Zelle 4 zu verwenden
if 'rpg_environment_instance' in locals() and isinstance(rpg_environment_instance, rpg_env.RPGEnv):
    # Prüfen ob die Instanz mit der neu geladenen Klasse übereinstimmt
    if type(rpg_environment_instance) == rpg_env.RPGEnv:
        env = rpg_environment_instance
        print("Verwende bestehende RPGEnv Instanz.")
    else:
        print("Bestehende Instanz ist vom alten Typ, erstelle neue Instanz...")
        env = None # Erzwinge Neuerstellung

if env is None:
    print("Erstelle neue RPGEnv Instanz für den Test...")
    try:
        # Sicherstellen, dass INITIALIZED_SKILLS aus Zelle 3 existiert
        if 'INITIALIZED_SKILLS' not in locals():
            raise NameError("INITIALIZED_SKILLS nicht gefunden. Bitte Zelle 3 ausführen.")

        env = rpg_env.RPGEnv(
            char_param_definitions=rpg_definitions.CHAR_PARAMS,
            all_buffs_debuffs_names=rpg_definitions.ALL_BUFFS_DEBUFFS_NAMES,
            max_buff_duration=rpg_definitions.MAX_BUFF_DURATION,
            max_stacks=rpg_definitions.MAX_STACKS,
            render_mode='ansi'
        )
        rpg_environment_instance = env # Speichere die neue Instanz global, falls sie wiederverwendet wird
        print("Neue RPGEnv Instanz erstellt und als 'rpg_environment_instance' gespeichert.")
    except Exception as e:
        print(f"FEHLER beim Erstellen der Test-RPGEnv Instanz: {e}")
        traceback.print_exc()
        raise e

# --- 3. Testparameter ---
hero_test_class = "Krieger" # Klasse zum Testen
# enemy_test_class = "Schurke" # Wird jetzt durch Env bestimmt
max_test_turns = 30       # Maximale Runden für diesen Test (ggf. erhöhen/senken)
# Wähle eine Aktion zum Testen (Basis-Angriff ist zuverlässig)
action_to_test = env._basic_attack_idx
action_name_to_test = env.action_to_name_map.get(action_to_test, "Unbekannt")

print(f"\nTest Setup: Held={hero_test_class}, Aktion={action_name_to_test} ({action_to_test}), Max Runden={max_test_turns}")

# --- 4. Testdurchführung ---
log_history = []
try:
    # Umgebung mit fester Heldenklasse zurücksetzen
    env.set_fixed_class(hero_test_class)
    obs, info = env.reset()

    # Überprüfe, ob Held und Gegner erstellt wurden
    if not env.hero or not env.enemy:
         raise RuntimeError("Held oder Gegner konnte im Reset nicht erstellt werden.")

    log_history.append(f"Testkampf beginnt: {env.hero.name} (Lvl {env.hero.level}) vs {env.enemy.name} (Lvl {env.enemy.level})")
    print(f"Startzustand: Held={str(env.hero)}")
    print(f"Startzustand: Gegner={str(env.enemy)}")

    # UI initial anzeigen
    clear_output(wait=True)
    display(rpg_ui.generate_html_ui(env, log_history))
    print(f"Runde 0: Kampf beginnt.")
    # time.sleep(1) # Kurze Pause zu Beginn

    terminated = False
    truncated = False
    turn = 0

    while not terminated and not truncated and turn < max_test_turns:
        turn += 1
        #print(f"\n--- Runde {turn} ---") # Weniger Output während des Laufs

        # Feste Aktion ausführen
        current_action = action_to_test

        # Umgebungsschritt
        obs, reward, terminated, truncated, info = env.step(current_action)

        # Log-Nachrichten hinzufügen (selektiv)
        action_name = env.action_to_name_map.get(current_action, "Unbekannt")
        log_entry = f"R {turn}: {env.hero.name} nutzt {action_name}."
        if info.get('last_reward_components', {}).get('Schaden verursacht Bonus', 0) > 0 :
             # Finde den tatsächlichen Schaden (ungefähre Schätzung aus Reward)
             # hp_diff = info['last_reward_components']['Schaden verursacht Bonus'] / env.DAMAGE_REWARD_FACTOR * env.enemy.max_hp
             # log_entry += f" ({env.enemy.name} erleidet Schaden)" # Vereinfacht
             pass # Weniger ausführlich im Log
        log_history.append(log_entry)

        # Wichtige Ereignisse loggen
        if info.get('last_reward_components', {}).get('Sieg', 0) > 0:
            log_history.append(f"R {turn}: {env.enemy.name} wurde besiegt!")
            print(f"Runde {turn}: Gegner besiegt!")
        if info.get('last_reward_components', {}).get('Niederlage', 0) < 0:
             log_history.append(f"R {turn}: {env.hero.name} wurde besiegt!")
             print(f"Runde {turn}: Held besiegt!")
        if info.get('last_reward_components', {}).get('Level Up Bonus', 0) > 0:
             level_up_msg = f"R {turn}: LEVEL UP für {env.hero.name} auf Level {env.hero.level}!"
             log_history.append(level_up_msg)
             print(level_up_msg)


        # Aktuellen Zustand und UI anzeigen (weniger häufig für Übersicht)
        if turn % 1 == 0 or terminated or truncated: # Update jede Runde oder am Ende
            clear_output(wait=True) # Löscht vorherige Ausgabe
            display(rpg_ui.generate_html_ui(env, log_history[-20:])) # Zeige die letzten 20 Log-Einträge
            print(f"Runde {turn}: Aktion={action_name}, Reward={reward:.2f}, Term={terminated}, Trunc={truncated}")
            # print(f"Info: {info}") # Info ist jetzt sehr lang, ggf. weglassen
            print(f"Held:  {str(env.hero)}")
            print(f"Gegner:{str(env.enemy)}")
            time.sleep(0.2) # Kurze Pause für die Anzeige

    # --- 5. Ergebnis prüfen ---
    print("\n--- Testlauf Kampfphase beendet ---")
    if env.hero:
        print(f"Finaler Heldenstatus nach Kampf: {str(env.hero)}")
        if terminated and not env.hero.is_dead() and hasattr(env.enemy, '_xp_granted') and env.enemy._xp_granted:
             print("Info: Gegner besiegt, XP sollten vergeben worden sein.")
        elif not terminated:
             print("Info: Kampf nicht normal beendet (Timeout/Max Turns).")

    if terminated:
        print("Kampf normal beendet.")
    if truncated:
        print("Kampf durch Timeout beendet.")
    if turn >= max_test_turns and not (terminated or truncated):
         print(f"Kampf nach maximalen Testrunden ({max_test_turns}) beendet.")

    # --- 6. Optionaler manueller Level-Up Test ---
    if env.hero and not env.hero.is_dead():
        print("\n--- Manueller Level-Up Test ---")
        # Berechne benötigte XP für nächsten Level Up
        xp_needed_for_next = env.hero.xp_to_next_level - env.hero.current_xp
        # Gebe genug XP + 1, um sicher aufzusteigen
        xp_to_add = max(1, xp_needed_for_next + 1)

        print(f"Aktuell: Lvl {env.hero.level}, XP {env.hero.current_xp}/{env.hero.xp_to_next_level}")
        print(f"Gebe {env.hero.name} manuell {xp_to_add} XP...")
        level_before_manual_xp = env.hero.level
        env.hero.gain_xp(xp_to_add) # Füge XP hinzu

        # Prüfe ob Level Up stattgefunden hat
        if env.hero.level > level_before_manual_xp:
            print(f"ERFOLG: Manueller Level Up hat stattgefunden! Neuer Level: {env.hero.level}")
            print(f"Neuer Status nach manuellem XP-Push: {str(env.hero)}")
            # Optional: Prüfe, ob Stats sich sinnvoll geändert haben (visuell)
        else:
            print("WARNUNG: Manueller Level Up hat NICHT stattgefunden? Überprüfe XP-Logik oder ob Held schon Max-Level ist.")
            print(f"Status nach manuellem XP-Push: {str(env.hero)}")
    elif env.hero and env.hero.is_dead():
         print("\nManueller Level-Up Test übersprungen (Held ist tot).")
    else:
        print("\nManueller Level-Up Test übersprungen (Held nicht vorhanden).")


except Exception as e:
    print(f"\nFEHLER während des Testlaufs: {e}")
    traceback.print_exc()

print("\n--- Testlauf Skript Ende ---")

Runde 7: Aktion=Basis-Angriff, Reward=19.27, Term=True, Trunc=False
Held:  Krieger_Hero (Krieger L:1) HP:135/145 XP:63/100 S:100/100 Atr:[S:15 G:8 K:12 I:5 W:5]
Gegner:Skelett_L2_979 (Skelett L:2) HP:0/70 M:30/30 Atr:[S:6 G:8 K:4 I:8 W:6]

--- Testlauf Kampfphase beendet ---
Finaler Heldenstatus nach Kampf: Krieger_Hero (Krieger L:1) HP:135/145 XP:63/100 S:100/100 Atr:[S:15 G:8 K:12 I:5 W:5]
Info: Gegner besiegt, XP sollten vergeben worden sein.
Kampf normal beendet.

--- Manueller Level-Up Test ---
Aktuell: Lvl 1, XP 63/100
Gebe Krieger_Hero manuell 38 XP...
LEVEL UP! Krieger_Hero ist jetzt Level 2!
Attribute erhöht: KON +1, STR +1
Krieger_Hero regeneriert beim Level Up (+15 HP).
Krieger_Hero Ressourcen aufgefüllt.
ERFOLG: Manueller Level Up hat stattgefunden! Neuer Level: 2
Neuer Status nach manuellem XP-Push: Krieger_Hero (Krieger L:2) HP:150/160 XP:1/200 S:100/100 Atr:[S:16 G:8 K:13 I:5 W:5]

--- Testlauf Skript Ende ---


# Code